# Lab 9 — Advanced Evaluation: Consistency, Cost-Quality & Fairness

**Prerequisite:** Lab 2 (Agent Evaluation). That lab taught *correctness* — does the answer pass?  
This lab adds the three dimensions you also need before shipping to **real users at scale**:

| Dimension | Question it answers | When it bites you |
|-----------|--------------------|-----------------|
| **Consistency** | Same input → same answer? | High-stakes agents (medical/legal/finance) that flip-flop |
| **Cost-quality Pareto** | Cheapest model that still passes? | A $20k/month bill for a task gpt-4o-mini could do |
| **Fairness** | Equal quality across user groups? | A bias headline / regulatory action |

> An interviewer who asks *"how would you evaluate an agent?"* and hears only *"LLM-as-judge"* is unimpressed. These three are how you show senior-level thinking.

> **Tested on:** `gpt-4o-mini` (+ one `gpt-4o` comparison in §2).  
> **Cost:** ~$0.05–0.15 for a full run. This lab makes **many** LLM calls — consistency runs the same prompt N times, Pareto runs every prompt across each model, and the fairness audit runs every template × every demographic variant **plus** a judge call each (≈15–20 calls). Reduce `n`, the prompt list, or the variant list to cut cost.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from advanced_evals import consistency_score, pareto_analysis, fairness_audit
from openai import OpenAI
client = OpenAI()
print('Lab 9 ready ✓')

## 1. Consistency — stability under repetition

Run the same prompt N times and measure how much the answers vary. The trick is that **the right consistency target depends on the task**:
- A medical triage agent should be near-1.0 (same symptoms → same advice).
- A brainstorming agent should be *low* — repeating the same idea 5 times is a bug, not a feature.

Temperature is your main lever. Let's see it directly.

In [ ]:
def make_agent(temperature: float):
    def agent(prompt: str) -> str:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature,
            max_tokens=120,
        )
        return resp.choices[0].message.content
    return agent

prompt = 'In one sentence, what is the most important quality in a software engineer?'

for temp in [0.0, 1.0]:
    cs = consistency_score(make_agent(temp), prompt, n=4)
    print(f'temperature={temp}: semantic_similarity={cs["semantic_similarity"]:.2f}  std_length={cs["std_length"]:.0f}')

You should see higher similarity at temperature 0. **Interview gotcha:** temperature 0 is *not* fully deterministic — GPU non-determinism and MoE routing still cause occasional drift. "Deterministic" is the wrong word; "low variance" is right.

## 2. Cost-quality Pareto — the cheapest model that clears your bar

Don't ask *"which model is best?"* Ask *"which is the cheapest that still passes my eval?"* That's the Pareto-optimal choice. We grade with a simple substring check here; in production you'd reuse the LLM-as-judge from Lab 2.

In [ ]:
prompts = [
    'What is the capital of Japan?',
    'What is 17 * 23?',
    'Name the largest planet in our solar system.',
]
answers = ['tokyo', '391', 'jupiter']

def grade(prompt, output):
    idx = prompts.index(prompt)
    return answers[idx] in output.lower()

# Compare a cheap and an expensive model on the same suite
rows = pareto_analysis(prompts, ['gpt-4o-mini', 'gpt-4o'], grade)
for r in rows:
    print(f"{r['model']:14} pass_rate={r['pass_rate']:.0%}  cost=${r['total_cost_usd']:.6f}")

print('\nRule: pick the cheapest model whose pass_rate >= your bar (e.g. 0.95).')

## 3. Fairness audit — counterfactual demographic testing

The method: take a request, change **only** a demographic signal (name, age, gender), and check whether answer *quality* stays equal. A large spread is a signal to investigate — not proof of bias (samples are tiny), but exactly the check a responsible-AI reviewer expects you to know.

In [ ]:
def career_agent(prompt: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'You are a helpful, even-handed career coach.'},
            {'role': 'user', 'content': prompt},
        ],
        max_tokens=250,
    )
    return resp.choices[0].message.content

audit = fairness_audit(career_agent, disparity_threshold=0.15)
print(f"Any dimension flagged: {audit['any_flagged']}\n")
for f in audit['findings']:
    flag = '⚠️ FLAGGED' if f['flagged'] else 'ok'
    print(f"[{f['dimension']:6}] spread={f['spread']:.2f}  {flag}")
    for variant, score in f['scores'].items():
        print(f'      {variant:20} {score:.2f}')

## Exercise

1. Add a **refusal-rate** check to the fairness audit: count how often the agent refuses or hedges per demographic group. A differential refusal rate is a subtler form of bias than quality spread.
2. Extend `pareto_analysis` to also record `avg_latency`, then decide: for a user-facing chat, would you pick the cheapest model or the fastest one that passes? Justify it as you would to a PM.

In [ ]:
# Your code here
